# SenPa-MAE Test

In [1]:
import sys
import os
import torch
import numpy as np
from pathlib import Path

# Set up paths
os.chdir(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

# Calculate the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Adding {project_root} to sys.path")

# Add to sys.path -> necessary for importing the src package
if project_root not in sys.path:
    sys.path.append(project_root)

# Import necessary modules
import hydra
from omegaconf import OmegaConf
from hydra.core.global_hydra import GlobalHydra
from hydra import initialize, compose
from geofm_src.factory import create_dataset, create_model, model_registry, dataset_registry

# Initialize Hydra
# Clear any previous initialization
GlobalHydra.instance().clear()
# Initialize with the correct config path
initialize(config_path="../geofm_src/configs/", caller_stack_depth=2)

# Print available models in the registry
print("Available models in the registry:")
for model_name in model_registry.keys():
    print(f"- {model_name}")

# Print available datasets in the registry
print("\nAvailable datasets in the registry:")
for dataset_name in dataset_registry.keys():
    print(f"- {dataset_name}")

Adding /home/ando to sys.path


/home/ando/.conda/envs/fm/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Available models in the registry:
- croma
- dinov2
- softcon
- dofa
- senpamae
- panopticon
- galileo
- anysat

Available datasets in the registry:
- geobench
- resisc45
- benv2
- corine
- digital_typhoon
- tropical_cyclone
- hyperview
- fmow
- dummy


/data/tmp/ipykernel_601499/2916192492.py:31: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../geofm_src/configs/", caller_stack_depth=2)


In [2]:
# Function to create a dummy dataset config
def create_dummy_dataset_config(num_channels=4, image_resolution=224, num_classes=19, task="classification", senpamae_channels=[0,1,2,3]):
    """Create a dummy dataset config for testing."""
    config = {
        "dataset_type": "dummy",
        "num_channels": num_channels,
        "image_resolution": image_resolution,
        "num_classes": num_classes,
        "task": task,
        "band_gsds": [10.0] * (num_channels),  # Ground sample distance for each band
        "senpamae_channels": senpamae_channels,  # For SenPaMAE model
        "senpamae_srf_name": "rfs_sentinel2_a.npy"  # For SenPaMAE model
    }
    return OmegaConf.create(config)

# Function to load a model config
def load_model_config(model_name):
    """Load a model config from the configs directory."""
    try:
        # Try to load the model config using Hydra
        model_config = compose(config_name=f"{model_name}")
        return model_config
    except Exception as e:
        print(f"Error loading model config: {e}")
        # Create a basic model config if loading fails
        config = {
            "model_type": 'senpamae',
            "embed_dim": 768,
            "image_resolution": 144,
            "pretrained_path": '/data/panopticon/other_model_ckpts/senpamae_doubleParameterEmbedding.pt',
            "num_channels": 4,
            "segm_blk_indices": [4,6,10,11],
            "accel_cls_blk_indices": [8, 9, 10, 11],
            "default_cls_blk_indices": [11],
            "default_feat_out_dim": 768
        }
        return OmegaConf.create(config)
    
# Function to instantiate and test a model
def test_model(model_name, dataset_config=None):
    """Instantiate a model and run a forward pass with dummy data."""
    print(f"\n--- Testing {model_name} model ---")
    
    # Load model config
    model_config = load_model_config(model_name)
    model_config.model_type = model_name
    
    # Adjust model config to match dataset
    # if dataset_config:
    #     model_config.num_channels = dataset_config.num_channels
    #     model_config.image_resolution = dataset_config.image_resolution
    
    print(f"Model config:")
    print(OmegaConf.to_yaml(model_config))
    
    # try:
        # Create model
    model = create_model(model_config, dataset_config)
    print(f"Successfully created {model_name} model")
    
    # Create dummy input
    batch_size = 5
    channels = dataset_config.num_channels
    resolution = dataset_config.image_resolution
    dummy_input = torch.rand(batch_size, channels, resolution, resolution)

    print(f"Dummy input shape: {dummy_input.shape}")
    
    # Get blocks (features)
    model.load_encoder(model_config.default_cls_blk_indices)
    blocks = model.get_blocks(dummy_input)
    print(f"Extracted {len(blocks)} feature blocks")
    
    # Get feature vector
    features = model.default_blocks_to_featurevec(blocks)
    print(f"Feature vector shape: {features.shape}")
    
    return model, features
    
    # except Exception as e:
    #     print(f"Error instantiating or running {model_name} model: {e}")
    #     return None, None

In [3]:
dummy_dataset_config = create_dummy_dataset_config(num_channels=4, image_resolution=144, senpamae_channels=[2,0,1,6])
print("\nDummy dataset config:")
print(OmegaConf.to_yaml(dummy_dataset_config))


Dummy dataset config:
dataset_type: dummy
num_channels: 4
image_resolution: 144
num_classes: 19
task: classification
band_gsds:
- 10.0
- 10.0
- 10.0
- 10.0
- 10.0
senpamae_channels:
- 2
- 0
- 1
- 6
senpamae_srf_name: rfs_sentinel2_a.npy



In [5]:
# Test a specific model (e.g., SenPaMAE)
model_name = "senpamae"  # Change this to test different models
model, features = test_model(model_name, dummy_dataset_config)



--- Testing senpamae model ---
Error loading model config: Primary config directory not found.
Check that the config directory '/home/ando/.conda/envs/fm/lib/python3.10/site-packages/IPython/geofm_src/configs' exists and readable
Model config:
model_type: senpamae
embed_dim: 768
image_resolution: 144
pretrained_path: /data/panopticon/other_model_ckpts/senpamae_doubleParameterEmbedding.pt
num_channels: 4
segm_blk_indices:
- 4
- 6
- 10
- 11
accel_cls_blk_indices:
- 8
- 9
- 10
- 11
default_cls_blk_indices:
- 11
default_feat_out_dim: 768

Successfully created senpamae model
Dummy input shape: torch.Size([5, 4, 144, 144])


AssertionError: Band GSDs size 5 not equal to 4 channels